# Day 5: nflverse data (NFL)

**Phase 1: Foundations & Infrastructure**

## Objective
nflverse data (NFL)

## Task
Install nfl_data_py. Explore play-by-play, roster, and schedule data. Document key fields: EPA, WPA, CPOE, air yards, etc. Download seasons 2020–2025. Save in data/raw/nfl/.

**Deliverable:** Notebook day005_nflverse_data_exploration.ipynb with documented schema + downloaded datasets.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



In [ ]:
pip install nfl_data_py

In [8]:
import nfl_data_py as nfl
import pandas as pd
import os

# Define the local directory path for saving raw NFL data
data_dir = "/Users/davidbazalduamendez/Documents/GitHub/100-days-Data-Sports-Challenge/data/raw/nfl"

# Create the directory safely
os.makedirs(data_dir, exist_ok=True)


In [ ]:
import ssl
# Use ssl to ve able to download unverified 
ssl._create_default_https_context = ssl._create_unverified_context

# Download a 2023 data 
pbp_2023 = nfl.import_pbp_data([2023])
print(pbp_2023.shape)
display(pbp_2023.head())

2023 done.
Downcasting floats.
(49665, 397)


,play_id,game_id,old_game_id_x,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,was_pressure,route,defense_man_zone_type,defense_coverage_type,offense_names,defense_names,offense_positions,defense_positions,offense_numbers,defense_numbers
0,1.0,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,39.0,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,False,,,None,Christian Holmes;Jartavius Martin;Casey Toohil...,Antonio Hamilton;Andre Chachere;Kris Boyd;Chri...,CB;CB;DE;DE;FS;MLB;MLB;OLB;RB;RB;TE,CB;CB;CB;CB;ILB;ILB;K;OLB;OLB;SS;WR,34;20;95;50;39;51;47;52;24;23;87,33;36;29;35;47;51;5;52;43;22;0
2,55.0,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,False,,,None,Nick Gates;Andrew Wylie;Sam Howell;Brian Robin...,Marco Wilson;Kei'Trel Clark;L.J. Collier;Jonat...,C;G;QB;RB;T;T;T;TE;TE;WR;WR,CB;CB;DE;DE;DT;FS;ILB;ILB;OLB;OLB;SS,63;71;14;8;72;77;76;82;87;17;1,20;13;91;93;95;34;7;10;25;52;3
3,77.0,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,False,HITCH/CURL,ZONE_COVERAGE,COVER_3,Nick Gates;Andrew Wylie;Sam Howell;Brian Robin...,Marco Wilson;Kei'Trel Clark;L.J. Collier;Jonat...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,CB;CB;DE;DE;FS;ILB;ILB;OLB;OLB;SS;SS,63;71;14;8;72;77;76;82;4;17;1,20;13;91;93;34;7;10;25;97;3;22
4,102.0,2023_01_ARI_WAS,2023091007,WAS,ARI,REG,1,WAS,home,ARI,...,False,,,None,Nick Gates;Andrew Wylie;Sam Howell;Chris Rodri...,Marco Wilson;Kei'Trel Clark;L.J. Collier;Jonat...,C;G;QB;RB;T;T;T;TE;WR;WR;WR,CB;CB;DE;DE;FS;ILB;ILB;OLB;OLB;SS;SS,63;71;14;23;72;77;76;82;4;17;1,20;13;91;93;34;7;10;25;97;3;22


In [12]:
pbp_2023['play_type'].unique()

array([None, 'kickoff', 'run', 'pass', 'punt', 'no_play', 'extra_point',
       'field_goal', 'qb_kneel', 'qb_spike'], dtype=object)

In [14]:
# Filtrate by offensive plays
ofensiva_2023 = pbp_2023[pbp_2023['play_type'].isin(['pass', 'run'])]

# Verify how many are
print(f"Offensive real plays: {ofensiva_2023.shape[0]}")

Offensive real plays: 35600


In [15]:
# Filter only pass and run plays
offense_2023_df = pbp_2023[pbp_2023['play_type'].isin(['pass', 'run'])]

# Group by team in possession and calculate mean EPA
team_epa_df = offense_2023_df.groupby('posteam')['epa'].mean()

# Sort results to see the leaders
top_offenses_df = team_epa_df.sort_values(ascending=False)

print("Top 5 Offenses (EPA per play in 2023):")
print(top_offenses_df.head())

Top 5 Offenses (EPA per play in 2023):
posteam
SF     0.165566
BUF    0.114988
DAL    0.102504
MIA    0.080431
DET    0.079049
Name: epa, dtype: float32


In [16]:
# Filter only passing plays
passes_2023_df = offense_2023_df[offense_2023_df['play_type'] == 'pass']

# Group by passer and calculate both average EPA and total pass count
qb_epa_df = passes_2023_df.groupby('passer_player_name').agg(
    epa_per_play=('epa', 'mean'),
    total_passes=('epa', 'count')
)

# Sort by EPA to see the raw leaders
top_qbs_raw_df = qb_epa_df.sort_values(by='epa_per_play', ascending=False)

# Display the top 10
display(top_qbs_raw_df.head(10))

,epa_per_play,total_passes
passer_player_name,,
B.Mann,4.209327,1
J.Reeves-Maybin,4.143003,1
J.Jennings,2.946513,1
T.Morstead,2.670900,1
L.Cooke,2.564376,1
T.Townsend,2.439641,1
S.Clifford,2.143627,1
D.London,1.844015,1
D.Singletary,1.842297,1


In [17]:
# Apply the minimum threshold of 50 passes
qualified_qbs_df = top_qbs_raw_df[top_qbs_raw_df['total_passes'] >= 50]

# Display the clean Top 10 most efficient QBs
print("Top 10 Most Efficient QBs in 2023 (Min. 50 passes):")
display(qualified_qbs_df.head(10))

Top 10 Most Efficient QBs in 2023 (Min. 50 passes):


,epa_per_play,total_passes
passer_player_name,,
B.Purdy,0.274436,580
J.Love,0.160159,665
D.Prescott,0.154885,695
T.Tagovailoa,0.150526,630
J.Goff,0.128745,754
J.Allen,0.122985,675
M.Stafford,0.108764,589
C.Stroud,0.107956,588
J.Browning,0.087135,268


In [18]:
# Filter for regular plays (pass and run)
defense_plays_df = pbp_2023[pbp_2023['play_type'].isin(['pass', 'run'])]

# Group by defensive team and calculate average EPA allowed
defensive_epa_df = defense_plays_df.groupby('defteam')['epa'].mean()

# Sort the results (ascending, because lower/negative EPA is better for defense)
top_defenses_df = defensive_epa_df.sort_values(ascending=True)

print("Top 5 Defenses (EPA allowed per play in 2023):")
display(top_defenses_df.head())

Top 5 Defenses (EPA allowed per play in 2023):


defteam
CLE   -0.169176
BAL   -0.142003
NYJ   -0.118892
NO    -0.099879
DAL   -0.084326
Name: epa, dtype: float32

In [19]:
# Group by defensive team and calculate total sacks
sacks_by_team_df = defense_plays_df.groupby('defteam')['sack'].sum()

# Sort from highest to lowest to see the most aggressive defenses
top_pass_rush_df = sacks_by_team_df.sort_values(ascending=False)

print("Top 5 Defenses (Total Sacks in 2023):")
display(top_pass_rush_df.head())

Top 5 Defenses (Total Sacks in 2023):


defteam
KC     64.0
BAL    62.0
MIA    56.0
BUF    55.0
HOU    53.0
Name: sack, dtype: float32